In [1]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [2]:
conn = sqlite3.connect("../bluestock_mf.db")

In [3]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

display(tables)

,name
0,dim_fund
1,sqlite_sequence
2,fact_nav
3,fact_transactions
4,fact_performance
5,dim_date


In [5]:
pd.read_sql("""
SELECT COUNT(*) AS total_rows
FROM fact_nav;
""", conn)

,total_rows
0,46600


In [6]:
nav = pd.read_sql("""
SELECT
    f.fund_name,
    d.full_date,
    n.nav
FROM fact_nav n
JOIN dim_fund f
    ON n.fund_id = f.fund_id
JOIN dim_date d
    ON n.date_id = d.date_id
ORDER BY f.fund_name, d.full_date;
""", conn)

nav["full_date"] = pd.to_datetime(nav["full_date"])

print(nav.shape)
nav.head()

(46600, 3)


,fund_name,full_date,nav
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-03,34.9539
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720


In [7]:
nav["daily_return"] = (
    nav.groupby("fund_name")["nav"]
       .pct_change()
)

nav.head(10)

,fund_name,full_date,nav,daily_return
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-03,34.9539,NaN
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026
5,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-10,34.2431,0.002081
6,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-11,33.5750,-0.019510
7,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-12,33.2889,-0.008521
8,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-13,33.4652,0.005296
9,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-14,32.6823,-0.023394


In [8]:
daily_returns = nav.dropna(subset=["daily_return"]).copy()

print(daily_returns.shape)
daily_returns.head()

(46560, 4)


,fund_name,full_date,nav,daily_return
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026
5,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-10,34.2431,0.002081


In [9]:
daily_returns["daily_return"].describe()

count    46560.000000
mean         0.000499
std          0.015021
min         -0.066480
25%         -0.009725
50%          0.000508
75%          0.010680
max          0.067686
Name: daily_return, dtype: float64

In [10]:
cagr = (
    nav.groupby("fund_name")
       .agg(
           start_nav=("nav", "first"),
           end_nav=("nav", "last"),
           start_date=("full_date", "first"),
           end_date=("full_date", "last")
       )
       .reset_index()
)

cagr.head()

,fund_name,start_nav,end_nav,start_date,end_date
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,34.9539,99.7436,2022-01-03,2026-06-19
1,Aditya Birla Sun Life Flexi Cap Fund 26,50.8751,69.9110,2022-01-03,2026-06-19
2,Aditya Birla Sun Life Small Cap Fund 5,47.6988,48.1743,2022-01-03,2026-06-19
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,69.8914,104.1759,2022-01-03,2026-06-19
4,Axis Mutual Fund ELSS Fund 34,43.6750,65.3044,2022-01-03,2026-06-19


In [11]:
cagr["years"] = (
    (cagr["end_date"] - cagr["start_date"]).dt.days
    / 365.25
)

cagr.head()

,fund_name,start_nav,end_nav,start_date,end_date,years
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,34.9539,99.7436,2022-01-03,2026-06-19,4.457221
1,Aditya Birla Sun Life Flexi Cap Fund 26,50.8751,69.9110,2022-01-03,2026-06-19,4.457221
2,Aditya Birla Sun Life Small Cap Fund 5,47.6988,48.1743,2022-01-03,2026-06-19,4.457221
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,69.8914,104.1759,2022-01-03,2026-06-19,4.457221
4,Axis Mutual Fund ELSS Fund 34,43.6750,65.3044,2022-01-03,2026-06-19,4.457221


In [12]:
cagr["CAGR"] = (
    (cagr["end_nav"] / cagr["start_nav"])
    ** (1 / cagr["years"])
    - 1
)

cagr.head()

,fund_name,start_nav,end_nav,start_date,end_date,years,CAGR
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,34.9539,99.7436,2022-01-03,2026-06-19,4.457221,0.265228
1,Aditya Birla Sun Life Flexi Cap Fund 26,50.8751,69.9110,2022-01-03,2026-06-19,4.457221,0.073915
2,Aditya Birla Sun Life Small Cap Fund 5,47.6988,48.1743,2022-01-03,2026-06-19,4.457221,0.002228
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,69.8914,104.1759,2022-01-03,2026-06-19,4.457221,0.093681
4,Axis Mutual Fund ELSS Fund 34,43.6750,65.3044,2022-01-03,2026-06-19,4.457221,0.094453


In [13]:
cagr = cagr.sort_values(
    "CAGR",
    ascending=False
)

cagr[["fund_name", "CAGR"]].head(10)

,fund_name,CAGR
5,Canara Robeco ELSS Fund 29,0.405619
31,SBI Mutual Fund Large Cap Fund 6,0.352872
22,ICICI Prudential Large Cap Fund 9,0.340967
10,DSP Mutual Fund Multi Cap Fund 31,0.296392
30,Nippon India Small Cap Fund 28,0.270074
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.265228
35,Tata Mutual Fund Hybrid Fund 23,0.252029
34,Tata Mutual Fund Flexi Cap Fund 11,0.246722
25,Mirae Asset Large Cap Fund 2,0.229103
18,HDFC Mutual Fund Large Cap Fund 25,0.214055


In [14]:
fig = px.bar(
    cagr.head(10),
    x="CAGR",
    y="fund_name",
    orientation="h",
    title="Top 10 Mutual Funds by CAGR",
    labels={
        "CAGR": "Annual Growth Rate",
        "fund_name": "Fund"
    }
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600
)

fig.show()

In [15]:
sharpe = (
    daily_returns
    .groupby("fund_name")
    .agg(
        mean_daily_return=("daily_return", "mean"),
        std_daily_return=("daily_return", "std")
    )
    .reset_index()
)

sharpe.head()

,fund_name,mean_daily_return,std_daily_return
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.001010,0.014739
1,Aditya Birla Sun Life Flexi Cap Fund 26,0.000379,0.014586
2,Aditya Birla Sun Life Small Cap Fund 5,0.000122,0.015077
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,0.000453,0.014814
4,Axis Mutual Fund ELSS Fund 34,0.000461,0.015166


In [16]:
risk_free_rate = 0.065
daily_rf = risk_free_rate / 252

daily_rf

0.00025793650793650796

In [17]:
sharpe["Sharpe_Ratio"] = (
    (sharpe["mean_daily_return"] - daily_rf)
    / sharpe["std_daily_return"]
) * np.sqrt(252)

sharpe.head()

,fund_name,mean_daily_return,std_daily_return,Sharpe_Ratio
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.001010,0.014739,0.809693
1,Aditya Birla Sun Life Flexi Cap Fund 26,0.000379,0.014586,0.132096
2,Aditya Birla Sun Life Small Cap Fund 5,0.000122,0.015077,-0.143042
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,0.000453,0.014814,0.208569
4,Axis Mutual Fund ELSS Fund 34,0.000461,0.015166,0.212154


In [18]:
sharpe = sharpe.sort_values(
    "Sharpe_Ratio",
    ascending=False
)

sharpe[["fund_name", "Sharpe_Ratio"]]

,fund_name,Sharpe_Ratio
5,Canara Robeco ELSS Fund 29,1.228868
22,ICICI Prudential Large Cap Fund 9,1.050101
31,SBI Mutual Fund Large Cap Fund 6,1.049914
10,DSP Mutual Fund Multi Cap Fund 31,0.896614
30,Nippon India Small Cap Fund 28,0.810713
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.809693
35,Tata Mutual Fund Hybrid Fund 23,0.754177
34,Tata Mutual Fund Flexi Cap Fund 11,0.732835
25,Mirae Asset Large Cap Fund 2,0.688279
18,HDFC Mutual Fund Large Cap Fund 25,0.637366


In [20]:
fig = px.bar(
    sharpe.head(10),
    x="Sharpe_Ratio",
    y="fund_name",
    orientation="h",
    title="Top 10 Funds by Sharpe Ratio",
    labels={
        "Sharpe_Ratio": "Sharpe Ratio",
        "fund_name": "Fund"
    }
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600
)

fig.show()

In [21]:
negative_returns = daily_returns[
    daily_returns["daily_return"] < 0
].copy()

print(negative_returns.shape)
negative_returns.head()

(22639, 4)


,fund_name,full_date,nav,daily_return
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026
6,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-11,33.5750,-0.019510
7,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-12,33.2889,-0.008521


In [22]:
downside = (
    negative_returns
    .groupby("fund_name")
    .agg(
        downside_std=("daily_return", "std")
    )
    .reset_index()
)

downside.head()

,fund_name,downside_std
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.008839
1,Aditya Birla Sun Life Flexi Cap Fund 26,0.008552
2,Aditya Birla Sun Life Small Cap Fund 5,0.009055
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,0.008569
4,Axis Mutual Fund ELSS Fund 34,0.009209


In [23]:
sortino = sharpe.merge(
    downside,
    on="fund_name",
    how="left"
)

sortino.head()

,fund_name,mean_daily_return,std_daily_return,Sharpe_Ratio,downside_std
0,Canara Robeco ELSS Fund 29,0.001417,0.014969,1.228868,0.009515
1,ICICI Prudential Large Cap Fund 9,0.001232,0.014730,1.050101,0.008592
2,SBI Mutual Fund Large Cap Fund 6,0.001276,0.015396,1.049914,0.008736
3,DSP Mutual Fund Multi Cap Fund 31,0.001107,0.015040,0.896614,0.008973
4,Nippon India Small Cap Fund 28,0.001030,0.015117,0.810713,0.008954


In [24]:
sortino["Sortino_Ratio"] = (
    (sortino["mean_daily_return"] - daily_rf)
    / sortino["downside_std"]
) * np.sqrt(252)

sortino.head()

,fund_name,mean_daily_return,std_daily_return,Sharpe_Ratio,downside_std,Sortino_Ratio
0,Canara Robeco ELSS Fund 29,0.001417,0.014969,1.228868,0.009515,1.933240
1,ICICI Prudential Large Cap Fund 9,0.001232,0.014730,1.050101,0.008592,1.800396
2,SBI Mutual Fund Large Cap Fund 6,0.001276,0.015396,1.049914,0.008736,1.850316
3,DSP Mutual Fund Multi Cap Fund 31,0.001107,0.015040,0.896614,0.008973,1.502833
4,Nippon India Small Cap Fund 28,0.001030,0.015117,0.810713,0.008954,1.368717


In [25]:
sortino = sortino.sort_values(
    "Sortino_Ratio",
    ascending=False
)

sortino[
    [
        "fund_name",
        "Sortino_Ratio"
    ]
]

,fund_name,Sortino_Ratio
0,Canara Robeco ELSS Fund 29,1.933240
2,SBI Mutual Fund Large Cap Fund 6,1.850316
1,ICICI Prudential Large Cap Fund 9,1.800396
3,DSP Mutual Fund Multi Cap Fund 31,1.502833
4,Nippon India Small Cap Fund 28,1.368717
5,Aditya Birla Sun Life Banking & PSU Debt Fund ...,1.350125
6,Tata Mutual Fund Hybrid Fund 23,1.335245
7,Tata Mutual Fund Flexi Cap Fund 11,1.273226
8,Mirae Asset Large Cap Fund 2,1.162125
9,HDFC Mutual Fund Large Cap Fund 25,1.109168


In [26]:
fig = px.bar(
    sortino.head(10),
    x="Sortino_Ratio",
    y="fund_name",
    orientation="h",
    title="Top 10 Mutual Funds by Sortino Ratio"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600
)

fig.show()

In [27]:
benchmark = (
    daily_returns
    .groupby("full_date")["daily_return"]
    .mean()
    .reset_index()
)

benchmark.rename(
    columns={"daily_return": "benchmark_return"},
    inplace=True
)

benchmark.head()

,full_date,benchmark_return
0,2022-01-04,-0.000341
1,2022-01-05,0.002583
2,2022-01-06,0.000880
3,2022-01-07,-0.001238
4,2022-01-10,0.000853


In [28]:
alpha_beta = daily_returns.merge(
    benchmark,
    on="full_date",
    how="left"
)

alpha_beta.head()

,fund_name,full_date,nav,daily_return,benchmark_return
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236,-0.000341
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870,0.002583
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752,0.000880
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026,-0.001238
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-10,34.2431,0.002081,0.000853


In [29]:
from scipy.stats import linregress

In [30]:
results = []

for fund in alpha_beta["fund_name"].unique():

    temp = alpha_beta[
        alpha_beta["fund_name"] == fund
    ]

    regression = linregress(
        temp["benchmark_return"],
        temp["daily_return"]
    )

    beta = regression.slope
    alpha = regression.intercept * 252

    results.append({
        "fund_name": fund,
        "Alpha": alpha,
        "Beta": beta
    })

alpha_beta_df = pd.DataFrame(results)

alpha_beta_df.head()

,fund_name,Alpha,Beta
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.113107,1.124444
1,Aditya Birla Sun Life Flexi Cap Fund 26,-0.001169,0.769765
2,Aditya Birla Sun Life Small Cap Fund 5,-0.099556,1.036810
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,-0.015032,1.026941
4,Axis Mutual Fund ELSS Fund 34,-0.022567,1.103025


In [31]:
alpha_beta_df = alpha_beta_df.sort_values(
    "Alpha",
    ascending=False
)

alpha_beta_df

,fund_name,Alpha,Beta
31,SBI Mutual Fund Large Cap Fund 6,0.211865,0.873059
5,Canara Robeco ELSS Fund 29,0.201715,1.235547
10,DSP Mutual Fund Multi Cap Fund 31,0.184441,0.752870
22,ICICI Prudential Large Cap Fund 9,0.161829,1.183212
29,Nippon India Small Cap Fund 15,0.135902,0.567773
33,SBI Small Cap Fund - Direct Plan - Growth,0.134952,0.611162
30,Nippon India Small Cap Fund 28,0.129175,1.037273
25,Mirae Asset Large Cap Fund 2,0.117716,0.866962
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,0.113107,1.124444
35,Tata Mutual Fund Hybrid Fund 23,0.106819,1.103885


In [32]:
fig = px.bar(
    alpha_beta_df.head(10),
    x="Alpha",
    y="fund_name",
    orientation="h",
    title="Top 10 Funds by Alpha"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600
)

fig.show()

In [33]:
drawdown = nav.copy()

drawdown["running_max"] = (
    drawdown
    .groupby("fund_name")["nav"]
    .cummax()
)

drawdown.head()

,fund_name,full_date,nav,daily_return,running_max
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-03,34.9539,NaN,34.9539
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236,34.9539
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870,34.9539
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752,34.9539
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026,34.9539


In [34]:
drawdown["drawdown"] = (
    drawdown["nav"] / drawdown["running_max"]
) - 1

drawdown.head()

,fund_name,full_date,nav,daily_return,running_max,drawdown
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-03,34.9539,NaN,34.9539,0.000000
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236,34.9539,-0.014236
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870,34.9539,-0.017065
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752,34.9539,-0.008463
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026,34.9539,-0.022369


In [35]:
max_drawdown = (
    drawdown
    .groupby("fund_name")
    .agg(
        Max_Drawdown=("drawdown", "min")
    )
    .reset_index()
)

max_drawdown.head()

,fund_name,Max_Drawdown
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,-0.219685
1,Aditya Birla Sun Life Flexi Cap Fund 26,-0.296953
2,Aditya Birla Sun Life Small Cap Fund 5,-0.411764
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,-0.437433
4,Axis Mutual Fund ELSS Fund 34,-0.306074


In [36]:
max_drawdown = max_drawdown.sort_values(
    "Max_Drawdown"
)

max_drawdown

,fund_name,Max_Drawdown
28,Nippon India Large Cap Fund - Direct Plan Grow...,-0.643702
23,Kotak Mutual Fund Debt Fund 22,-0.589650
9,DSP Mutual Fund Hybrid Fund 27,-0.579795
38,UTI Mutual Fund Index Fund 14,-0.547543
16,HDFC Money Market Fund - Growth Option - Direc...,-0.514860
14,Franklin Templeton Index Fund 24,-0.511199
8,DSP Mutual Fund ELSS Fund 1,-0.501215
36,UTI Mutual Fund Flexi Cap Fund 18,-0.461845
39,quant Mid Cap Fund - Growth Option - Direct Plan,-0.449913
3,Axis ELSS Tax Saver Fund - Direct Plan - Growt...,-0.437433


In [37]:
fig = px.bar(
    max_drawdown.head(10),
    x="Max_Drawdown",
    y="fund_name",
    orientation="h",
    title="Top 10 Funds by Maximum Drawdown"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600
)

fig.show()

In [41]:
performance = pd.read_sql("""
SELECT
    p.*,
    f.fund_name
FROM fact_performance p
JOIN dim_fund f
ON p.fund_id = f.fund_id
""", conn)

print(performance.shape)
performance.head()

(40, 8)


,performance_id,fund_id,return_1y,return_3y,return_5y,expense_ratio,risk_grade,fund_name
0,1,1,0.23,19.87,16.48,2.01,Low,SBI Small Cap Fund - Direct Plan - Growth
1,2,2,26.58,11.47,9.41,0.28,Low,Aditya Birla Sun Life Banking & PSU Debt Fund ...
2,3,3,22.14,17.24,18.13,1.28,Very High,Axis ELSS Tax Saver Fund - Direct Plan - Growt...
3,4,4,29.13,23.42,13.23,0.53,High,Nippon India Large Cap Fund - Direct Plan Grow...
4,5,5,13.68,11.09,5.64,2.11,High,HDFC Money Market Fund - Growth Option - Direc...


In [42]:
scorecard = (
    cagr[["fund_name", "CAGR"]]
    .merge(
        sharpe[["fund_name", "Sharpe_Ratio"]],
        on="fund_name"
    )
    .merge(
        sortino[["fund_name", "Sortino_Ratio"]],
        on="fund_name"
    )
    .merge(
        alpha_beta_df[["fund_name", "Alpha", "Beta"]],
        on="fund_name"
    )
    .merge(
        max_drawdown,
        on="fund_name"
    )
    .merge(
        performance[
            [
                "fund_name",
                "expense_ratio"
            ]
        ],
        on="fund_name"
    )
)

scorecard.head()

,fund_name,CAGR,Sharpe_Ratio,Sortino_Ratio,Alpha,Beta,Max_Drawdown,expense_ratio
0,Canara Robeco ELSS Fund 29,0.405619,1.228868,1.933240,0.201715,1.235547,-0.216997,1.17
1,SBI Mutual Fund Large Cap Fund 6,0.352872,1.049914,1.850316,0.211865,0.873059,-0.212917,1.18
2,ICICI Prudential Large Cap Fund 9,0.340967,1.050101,1.800396,0.161829,1.183212,-0.287348,1.38
3,DSP Mutual Fund Multi Cap Fund 31,0.296392,0.896614,1.502833,0.184441,0.752870,-0.245620,1.76
4,Nippon India Small Cap Fund 28,0.270074,0.810713,1.368717,0.129175,1.037273,-0.284496,2.05


In [43]:
scorecard["CAGR_Rank"] = scorecard["CAGR"].rank(ascending=False)

scorecard["Sharpe_Rank"] = scorecard["Sharpe_Ratio"].rank(ascending=False)

scorecard["Alpha_Rank"] = scorecard["Alpha"].rank(ascending=False)

scorecard["Expense_Rank"] = scorecard["expense_ratio"].rank()

scorecard["Drawdown_Rank"] = scorecard["Max_Drawdown"].rank()

In [44]:
scorecard["Score"] = (
    (41 - scorecard["CAGR_Rank"]) * 0.30 +
    (41 - scorecard["Sharpe_Rank"]) * 0.25 +
    (41 - scorecard["Alpha_Rank"]) * 0.20 +
    (41 - scorecard["Expense_Rank"]) * 0.15 +
    (41 - scorecard["Drawdown_Rank"]) * 0.10
)

scorecard["Score"] = (
    scorecard["Score"] /
    scorecard["Score"].max()
) * 100

In [45]:
scorecard = scorecard.sort_values(
    "Score",
    ascending=False
)

scorecard[
    [
        "fund_name",
        "Score"
    ]
]

,fund_name,Score
0,Canara Robeco ELSS Fund 29,100.000000
1,SBI Mutual Fund Large Cap Fund 6,97.458894
2,ICICI Prudential Large Cap Fund 9,96.711510
5,Aditya Birla Sun Life Banking & PSU Debt Fund ...,95.067265
11,Nippon India Small Cap Fund 15,94.618834
3,DSP Mutual Fund Multi Cap Fund 31,90.284006
4,Nippon India Small Cap Fund 28,85.351271
6,Tata Mutual Fund Hybrid Fund 23,84.753363
7,Tata Mutual Fund Flexi Cap Fund 11,83.408072
8,Mirae Asset Large Cap Fund 2,82.959641


In [46]:
fig = px.bar(
    scorecard.head(10),
    x="Score",
    y="fund_name",
    orientation="h",
    title="Top 10 Mutual Funds by Composite Score"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=600
)

fig.show()

In [47]:
scorecard.to_csv(
    "../reports/fund_scorecard.csv",
    index=False
)

print("fund_scorecard.csv exported successfully!")

fund_scorecard.csv exported successfully!


In [48]:
alpha_beta_df.to_csv(
    "../reports/alpha_beta.csv",
    index=False
)

print("alpha_beta.csv exported successfully!")

alpha_beta.csv exported successfully!


In [49]:
benchmark = (
    nav.groupby("full_date")["nav"]
       .mean()
       .reset_index()
)

benchmark.rename(
    columns={"nav": "Benchmark_NAV"},
    inplace=True
)

benchmark.head()

,full_date,Benchmark_NAV
0,2022-01-03,53.154093
1,2022-01-04,53.251387
2,2022-01-05,53.387275
3,2022-01-06,53.432263
4,2022-01-07,53.271622


In [50]:
top5 = scorecard.head(5)["fund_name"].tolist()

top5_nav = nav[
    nav["fund_name"].isin(top5)
].copy()

top5_nav.head()

,fund_name,full_date,nav,daily_return
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-03,34.9539,NaN
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026


In [51]:
benchmark_chart = top5_nav.merge(
    benchmark,
    on="full_date",
    how="left"
)

benchmark_chart.head()

,fund_name,full_date,nav,daily_return,Benchmark_NAV
0,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-03,34.9539,NaN,53.154093
1,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-04,34.4563,-0.014236,53.251387
2,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-05,34.3574,-0.002870,53.387275
3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-06,34.6581,0.008752,53.432263
4,Aditya Birla Sun Life Banking & PSU Debt Fund ...,2022-01-07,34.1720,-0.014026,53.271622


In [52]:
fig = px.line(
    benchmark_chart,
    x="full_date",
    y="nav",
    color="fund_name",
    title="Top 5 Funds vs Benchmark"
)

fig.add_scatter(
    x=benchmark["full_date"],
    y=benchmark["Benchmark_NAV"],
    mode="lines",
    name="Benchmark",
    line=dict(width=4, dash="dash")
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="NAV",
    height=650
)

fig.show()